This notebook trains baseline models on both the men's and women's tournament training data.

In [24]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, log_loss, brier_score_loss

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

Load the data

In [25]:
men_df = pd.read_csv("../data/processed/m_tournament_training_dataset_advanced.csv")
women_df = pd.read_csv("../data/processed/w_tournament_training_dataset_advanced.csv")


Function to calculate the metrics

In [26]:
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5, print_results=True):
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        # convert scores to 0-1 range approximately
        y_prob = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        raise ValueError("Model does not support predict_proba or decision_function.")

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    results = {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "classification_report": report,
        "confusion_matrix": cm,
        "log_loss": logloss,
        "brier_score": brier,
        "auc": auc
    }

    if print_results:
        print("=== Classification Metrics ===")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        print("\n=== Classification Report ===")
        print(report)

        print("=== Confusion Matrix ===")
        print(cm)

        print("\n=== Probability Metrics ===")
        print(f"Log Loss:   {logloss:.4f}")
        print(f"Brier Score:{brier:.4f}")
        print(f"AUC:        {auc:.4f}")

    return results

### Helper function to store the model results

In [27]:
model_results = []

def add_model_result(model_results, dataset_name, model_name, results):
    model_results.append({
        "Dataset": dataset_name,
        "Model": model_name,
        "Accuracy": results["accuracy"],
        "F1": results["f1_score"],
        "Precision": results["precision"],
        "Recall": results["recall"],
        "LogLoss": results["log_loss"],
        "BrierScore": results["brier_score"],
        "AUC": results["auc"]
    })

In [28]:
season_cutoff = 2023
drop_cols = ["Season", "Team1ID", "Team2ID", "Target"]

In [29]:
men_train_df = men_df[men_df["Season"] < season_cutoff].copy()
men_test_df = men_df[men_df["Season"] >= season_cutoff].copy()

In [30]:
X_train_men = men_train_df.drop(columns=drop_cols)
y_train_men = men_train_df["Target"]
X_test_men = men_test_df.drop(columns=drop_cols)
y_test_men = men_test_df["Target"]

In [31]:
women_train_df = women_df[women_df["Season"] < season_cutoff].copy()
women_test_df = women_df[women_df["Season"] >= season_cutoff].copy()

In [32]:
X_train_women = women_train_df.drop(columns=drop_cols)
y_train_women = women_train_df["Target"]
X_test_women = women_test_df.drop(columns=drop_cols)
y_test_women = women_test_df["Target"]

### Models

Source: https://scikit-learn.org/stable/supervised_learning.html

- Classification Task
    - Logistic Regression
    - SVM
    - Decision Tree
    - Random Forest
    - Voting Classifier
    - XGBoost / AdaBoost
    - MLP Classifier (https://scikit-learn.org/stable/modules/neural_networks_supervised.html#classification)

### Logistic Regression

In [33]:
def run_logistic_regression(X_train, y_train, X_test, y_test):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### Decision Tree

In [34]:
def run_decision_tree(X_train, y_train, X_test, y_test):
    model = DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    )
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### Random Forest

In [35]:
def run_random_forest(X_train, y_train, X_test, y_test):
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### SVM Classifier

In [36]:
def run_svm(X_train, y_train, X_test, y_test):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### XGBoost

In [37]:
def run_xgboost(X_train, y_train, X_test, y_test):
    model = XGBClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### Ada Boost

In [38]:
def run_adaboost(X_train, y_train, X_test, y_test):
    model = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### MLP Classifier

In [39]:
def run_mlp(X_train, y_train, X_test, y_test):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### Voting Classifier

In [40]:
def run_voting_classifier(X_train, y_train, X_test, y_test):
    model = VotingClassifier(
        estimators=[
            ("xgb", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42
            )),
            ("rf", RandomForestClassifier(
                n_estimators=300,
                max_depth=5,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )),
            ("ada", AdaBoostClassifier(
                n_estimators=200,
                learning_rate=0.05,
                random_state=42
            ))
        ],
        voting="soft"
    )
    model.fit(X_train, y_train)
    results = evaluate_binary_classifier(model, X_test, y_test)
    return model, results

### Men baseline models

In [41]:
men_model_results = []

In [42]:
print("\nMen - Logistic Regression")
men_logistic_model, men_logistic_results = run_logistic_regression(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "Logistic Regression", men_logistic_results)


Men - Logistic Regression
=== Classification Metrics ===
Accuracy:   0.7015
F1 Score:   0.7015
Precision:  0.7015
Recall:     0.7015

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.70      0.70      0.70       201
           1       0.70      0.70      0.70       201

    accuracy                           0.70       402
   macro avg       0.70      0.70      0.70       402
weighted avg       0.70      0.70      0.70       402

=== Confusion Matrix ===
[[141  60]
 [ 60 141]]

=== Probability Metrics ===
Log Loss:   0.5758
Brier Score:0.1973
AUC:        0.7639


In [43]:
print("\nMen - Decision Tree")
men_decision_tree_model, men_decision_tree_results = run_decision_tree(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "Decision Tree", men_decision_tree_results)


Men - Decision Tree
=== Classification Metrics ===
Accuracy:   0.7189
F1 Score:   0.7224
Precision:  0.7136
Recall:     0.7313

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.72      0.71      0.72       201
           1       0.71      0.73      0.72       201

    accuracy                           0.72       402
   macro avg       0.72      0.72      0.72       402
weighted avg       0.72      0.72      0.72       402

=== Confusion Matrix ===
[[142  59]
 [ 54 147]]

=== Probability Metrics ===
Log Loss:   0.5738
Brier Score:0.1955
AUC:        0.7725


In [44]:
print("\nMen - Random Forest")
men_random_forest_model, men_random_forest_results = run_random_forest(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "Random Forest", men_random_forest_results)


Men - Random Forest
=== Classification Metrics ===
Accuracy:   0.7239
F1 Score:   0.7273
Precision:  0.7184
Recall:     0.7363

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.71      0.72       201
           1       0.72      0.74      0.73       201

    accuracy                           0.72       402
   macro avg       0.72      0.72      0.72       402
weighted avg       0.72      0.72      0.72       402

=== Confusion Matrix ===
[[143  58]
 [ 53 148]]

=== Probability Metrics ===
Log Loss:   0.5486
Brier Score:0.1849
AUC:        0.7972


In [45]:
print("\nMen - SVM")
men_svm_model, men_svm_results = run_svm(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "SVM", men_svm_results)


Men - SVM
=== Classification Metrics ===
Accuracy:   0.7338
F1 Score:   0.7318
Precision:  0.7374
Recall:     0.7264

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.74      0.74       201
           1       0.74      0.73      0.73       201

    accuracy                           0.73       402
   macro avg       0.73      0.73      0.73       402
weighted avg       0.73      0.73      0.73       402

=== Confusion Matrix ===
[[149  52]
 [ 55 146]]

=== Probability Metrics ===
Log Loss:   0.5659
Brier Score:0.1900
AUC:        0.7768


In [46]:
print("\nMen - XGBoost")
men_xgb_model, men_xgb_results = run_xgboost(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "XGBoost", men_xgb_results)


Men - XGBoost
=== Classification Metrics ===
Accuracy:   0.7015
F1 Score:   0.7030
Precision:  0.6995
Recall:     0.7065

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.70      0.70      0.70       201
           1       0.70      0.71      0.70       201

    accuracy                           0.70       402
   macro avg       0.70      0.70      0.70       402
weighted avg       0.70      0.70      0.70       402

=== Confusion Matrix ===
[[140  61]
 [ 59 142]]

=== Probability Metrics ===
Log Loss:   0.5819
Brier Score:0.1978
AUC:        0.7746


In [47]:
print("\nMen - AdaBoost")
men_adaboost_model, men_adaboost_results = run_adaboost(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "AdaBoost", men_adaboost_results)


Men - AdaBoost
=== Classification Metrics ===
Accuracy:   0.7488
F1 Score:   0.7612
Precision:  0.7252
Recall:     0.8010

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.78      0.70      0.73       201
           1       0.73      0.80      0.76       201

    accuracy                           0.75       402
   macro avg       0.75      0.75      0.75       402
weighted avg       0.75      0.75      0.75       402

=== Confusion Matrix ===
[[140  61]
 [ 40 161]]

=== Probability Metrics ===
Log Loss:   0.5519
Brier Score:0.1852
AUC:        0.8072


In [48]:
print("\nMen - MLP Classifier")
men_mlp_model, men_mlp_results = run_mlp(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "MLP Classifier", men_mlp_results)


Men - MLP Classifier
=== Classification Metrics ===
Accuracy:   0.6741
F1 Score:   0.6580
Precision:  0.6923
Recall:     0.6269

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.66      0.72      0.69       201
           1       0.69      0.63      0.66       201

    accuracy                           0.67       402
   macro avg       0.68      0.67      0.67       402
weighted avg       0.68      0.67      0.67       402

=== Confusion Matrix ===
[[145  56]
 [ 75 126]]

=== Probability Metrics ===
Log Loss:   0.5794
Brier Score:0.1990
AUC:        0.7625


In [49]:
print("\nMen - Voting Classifier")
men_voting_model, men_voting_results = run_voting_classifier(
    X_train_men, y_train_men, X_test_men, y_test_men
)
add_model_result(men_model_results, "Men", "Voting Classifier", men_voting_results)


Men - Voting Classifier
=== Classification Metrics ===
Accuracy:   0.7264
F1 Score:   0.7291
Precision:  0.7220
Recall:     0.7363

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.72      0.72       201
           1       0.72      0.74      0.73       201

    accuracy                           0.73       402
   macro avg       0.73      0.73      0.73       402
weighted avg       0.73      0.73      0.73       402

=== Confusion Matrix ===
[[144  57]
 [ 53 148]]

=== Probability Metrics ===
Log Loss:   0.5454
Brier Score:0.1836
AUC:        0.8000


In [50]:
men_results_df = pd.DataFrame(men_model_results)
men_results_df = men_results_df.sort_values(
    by=["LogLoss", "BrierScore", "AUC"],
    ascending=[True, True, False]
).reset_index(drop=True)

### Women baseline models

In [51]:
women_model_results = []

In [52]:
print("\nWomen - Logistic Regression")
women_logistic_model, women_logistic_results = run_logistic_regression(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "Logistic Regression", women_logistic_results)


Women - Logistic Regression
=== Classification Metrics ===
Accuracy:   0.7960
F1 Score:   0.7970
Precision:  0.7931
Recall:     0.8010

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.80      0.79      0.80       201
           1       0.79      0.80      0.80       201

    accuracy                           0.80       402
   macro avg       0.80      0.80      0.80       402
weighted avg       0.80      0.80      0.80       402

=== Confusion Matrix ===
[[159  42]
 [ 40 161]]

=== Probability Metrics ===
Log Loss:   0.4196
Brier Score:0.1419
AUC:        0.8799


In [53]:
print("\nWomen - Decision Tree")
women_decision_tree_model, women_decision_tree_results = run_decision_tree(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "Decision Tree", women_decision_tree_results)


Women - Decision Tree
=== Classification Metrics ===
Accuracy:   0.7910
F1 Score:   0.7910
Precision:  0.7910
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[159  42]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.6179
Brier Score:0.1491
AUC:        0.8700


In [54]:
print("\nWomen - Random Forest")
women_random_forest_model, women_random_forest_results = run_random_forest(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "Random Forest", women_random_forest_results)


Women - Random Forest
=== Classification Metrics ===
Accuracy:   0.7935
F1 Score:   0.7930
Precision:  0.7950
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.80      0.79       201
           1       0.80      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[160  41]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4151
Brier Score:0.1372
AUC:        0.8877


In [55]:
print("\nWomen - SVM")
women_svm_model, women_svm_results = run_svm(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "SVM", women_svm_results)


Women - SVM
=== Classification Metrics ===
Accuracy:   0.7886
F1 Score:   0.7891
Precision:  0.7871
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[158  43]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4884
Brier Score:0.1577
AUC:        0.8569


In [56]:
print("\nWomen - XGBoost")
women_xgb_model, women_xgb_results = run_xgboost(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "XGBoost", women_xgb_results)


Women - XGBoost
=== Classification Metrics ===
Accuracy:   0.7811
F1 Score:   0.7778
Precision:  0.7897
Recall:     0.7662

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.77      0.80      0.78       201
           1       0.79      0.77      0.78       201

    accuracy                           0.78       402
   macro avg       0.78      0.78      0.78       402
weighted avg       0.78      0.78      0.78       402

=== Confusion Matrix ===
[[160  41]
 [ 47 154]]

=== Probability Metrics ===
Log Loss:   0.4686
Brier Score:0.1535
AUC:        0.8706


In [57]:
print("\nWomen - AdaBoost")
women_adaboost_model, women_adaboost_results = run_adaboost(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "AdaBoost", women_adaboost_results)


Women - AdaBoost
=== Classification Metrics ===
Accuracy:   0.7910
F1 Score:   0.7910
Precision:  0.7910
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       201
           1       0.79      0.79      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[159  42]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4670
Brier Score:0.1513
AUC:        0.8811


In [58]:
print("\nWomen - MLP Classifier")
women_mlp_model, women_mlp_results = run_mlp(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "MLP Classifier", women_mlp_results)


Women - MLP Classifier
=== Classification Metrics ===
Accuracy:   0.7985
F1 Score:   0.7970
Precision:  0.8030
Recall:     0.7910

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       201
           1       0.80      0.79      0.80       201

    accuracy                           0.80       402
   macro avg       0.80      0.80      0.80       402
weighted avg       0.80      0.80      0.80       402

=== Confusion Matrix ===
[[162  39]
 [ 42 159]]

=== Probability Metrics ===
Log Loss:   0.4120
Brier Score:0.1357
AUC:        0.8919


In [59]:
print("\nWomen - Voting Classifier")
women_voting_model, women_voting_results = run_voting_classifier(
    X_train_women, y_train_women, X_test_women, y_test_women
)
add_model_result(women_model_results, "Women", "Voting Classifier", women_voting_results)


Women - Voting Classifier
=== Classification Metrics ===
Accuracy:   0.7861
F1 Score:   0.7850
Precision:  0.7889
Recall:     0.7811

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       201
           1       0.79      0.78      0.79       201

    accuracy                           0.79       402
   macro avg       0.79      0.79      0.79       402
weighted avg       0.79      0.79      0.79       402

=== Confusion Matrix ===
[[159  42]
 [ 44 157]]

=== Probability Metrics ===
Log Loss:   0.4255
Brier Score:0.1397
AUC:        0.8847


In [60]:
women_results_df = pd.DataFrame(women_model_results)
women_results_df = women_results_df.sort_values(
    by=["LogLoss", "BrierScore", "AUC"],
    ascending=[True, True, False]
).reset_index(drop=True)

Based on your baseline models:
- Decision Tree
- MLP
- SVM
- AdaBoost

are underperforming.

Our best models are:
- Voting classifier
- Random Forest
- XGBoost

### Feature importance

In [61]:
men_rf_importance_df = pd.DataFrame({
    "Feature": X_train_men.columns,
    "Importance": men_random_forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [62]:
men_xgb_importance_df = pd.DataFrame({
    "Feature": X_train_men.columns,
    "Importance": men_xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [63]:
women_rf_importance_df = pd.DataFrame({
    "Feature": X_train_women.columns,
    "Importance": women_random_forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [64]:
women_xgb_importance_df = pd.DataFrame({
    "Feature": X_train_women.columns,
    "Importance": women_xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [65]:
men_results_df

,Dataset,Model,Accuracy,F1,Precision,Recall,LogLoss,BrierScore,AUC
0,Men,Voting Classifier,0.726368,0.729064,0.721951,0.736318,0.545420,0.183585,0.800005
1,Men,Random Forest,0.723881,0.727273,0.718447,0.736318,0.548629,0.184906,0.797233
2,Men,AdaBoost,0.748756,0.761229,0.725225,0.800995,0.551948,0.185191,0.807158
3,Men,SVM,0.733831,0.731830,0.737374,0.726368,0.565858,0.189980,0.776850
4,Men,Decision Tree,0.718905,0.722359,0.713592,0.731343,0.573776,0.195506,0.772530
5,Men,Logistic Regression,0.701493,0.701493,0.701493,0.701493,0.575770,0.197311,0.763941
6,Men,MLP Classifier,0.674129,0.657963,0.692308,0.626866,0.579378,0.198983,0.762531
7,Men,XGBoost,0.701493,0.702970,0.699507,0.706468,0.581889,0.197771,0.774634


In [66]:
women_results_df

,Dataset,Model,Accuracy,F1,Precision,Recall,LogLoss,BrierScore,AUC
0,Women,MLP Classifier,0.798507,0.796992,0.803030,0.791045,0.411966,0.135737,0.891884
1,Women,Random Forest,0.793532,0.793017,0.795000,0.791045,0.415087,0.137167,0.887701
2,Women,Logistic Regression,0.796020,0.797030,0.793103,0.800995,0.419564,0.141882,0.879929
3,Women,Voting Classifier,0.786070,0.785000,0.788945,0.781095,0.425506,0.139746,0.884706
4,Women,AdaBoost,0.791045,0.791045,0.791045,0.791045,0.467004,0.151310,0.881092
5,Women,XGBoost,0.781095,0.777778,0.789744,0.766169,0.468588,0.153492,0.870647
6,Women,SVM,0.788557,0.789082,0.787129,0.791045,0.488351,0.157702,0.856897
7,Women,Decision Tree,0.791045,0.791045,0.791045,0.791045,0.617917,0.149067,0.869954


In [67]:
print("\nMen - Random Forest Feature Importances")
print(men_rf_importance_df.head(20))


Men - Random Forest Feature Importances
                         Feature  Importance
0                    SeedNumDiff    0.142037
1              SeedAdvantageFlag    0.093615
2                 AdvantageCount    0.078824
3                    RankingDiff    0.067857
4                     MarginDiff    0.055970
5                  NetRatingDiff    0.053830
6                      OffDefGap    0.050401
7                 AbsSeedNumDiff    0.027269
8           RankingAdvantageFlag    0.023884
9            SeedNumDiff_Squared    0.021749
10         SeedAdjustedDominance    0.019887
11                    OffEffDiff    0.018498
12      RankingAdjustedNetRating    0.017153
13                    WinPctDiff    0.016385
14    Margin_Ranking_Interaction    0.016103
15  NetRating_Margin_Interaction    0.015323
16        NetRatingAdvantageFlag    0.014999
17              AbsNetRatingDiff    0.014268
18                 AbsMarginDiff    0.014055
19          ReboundTurnoverCombo    0.013874


In [68]:
print("\nMen - XGBoost Feature Importances")
print(men_xgb_importance_df.head(20))



Men - XGBoost Feature Importances
                         Feature  Importance
0              SeedAdvantageFlag    0.327394
1                    SeedNumDiff    0.094515
2                 AdvantageCount    0.047057
3                      OffDefGap    0.021843
4                  NetRatingDiff    0.020391
5          OffEff_DefEff_Product    0.017584
6                    RankingDiff    0.016040
7                 AbsRankingDiff    0.015746
8       RankingAdjustedNetRating    0.014701
9            SeedNumDiff_Squared    0.014637
10                DominanceScore    0.014317
11         NetRatingDiff_Squared    0.013954
12         SeedAdjustedDominance    0.013935
13                 AbsMarginDiff    0.013925
14           RankingDiff_Squared    0.013862
15                  ControlScore    0.013660
16  PossessionControlInteraction    0.013509
17            TurnoverMarginDiff    0.013458
18          Three_FT_Interaction    0.012859
19    Margin_Ranking_Interaction    0.012846


In [69]:
print("\nWomen - Random Forest Feature Importances")
print(women_rf_importance_df.head(20))


Women - Random Forest Feature Importances
                       Feature  Importance
0                  RankingDiff    0.195285
1         RankingAdvantageFlag    0.131818
2                  SeedNumDiff    0.097516
3            SeedAdvantageFlag    0.074296
4               AbsRankingDiff    0.039244
5                   MarginDiff    0.039055
6   Margin_Ranking_Interaction    0.035342
7          RankingDiff_Squared    0.032687
8                    FGPctDiff    0.030513
9        Seed_Rank_Interaction    0.027592
10                   OffDefGap    0.026819
11               NetRatingDiff    0.024497
12        ReboundTurnoverCombo    0.018184
13    RankingAdjustedNetRating    0.017326
14                  OffEffDiff    0.014253
15       SeedAdjustedDominance    0.013308
16      NetRatingAdvantageFlag    0.012451
17              AbsSeedNumDiff    0.010393
18         SeedNumDiff_Squared    0.010389
19         ReboundControlScore    0.009238


In [70]:
print("\nWomen - XGBoost Feature Importances")
print(women_xgb_importance_df.head(20))


Women - XGBoost Feature Importances
                         Feature  Importance
0           RankingAdvantageFlag    0.402473
1                    RankingDiff    0.095667
2            RankingDiff_Squared    0.026083
3                 AbsRankingDiff    0.023722
4          Seed_Rank_Interaction    0.017760
5                 AdvantageCount    0.017054
6                  AbsMarginDiff    0.015835
7                    SeedNumDiff    0.015091
8          NetRatingDiff_Squared    0.014542
9            SeedNumDiff_Squared    0.014515
10          Three_FT_Interaction    0.014410
11                DominanceScore    0.013484
12                     OffDefGap    0.013468
13                AbsSeedNumDiff    0.013300
14  WinPct_NetRating_Interaction    0.012757
15                 NetRatingDiff    0.012540
16      RankingAdjustedNetRating    0.012135
17          FG_Three_Interaction    0.011658
18                     FGPctDiff    0.011490
19  NetRating_Margin_Interaction    0.011489


### Save output


In [71]:
men_results_df.to_csv("../data/processed/model_comparison_results_baseline_men.csv", index=False)
women_results_df.to_csv("../data/processed/model_comparison_results_baseline_women.csv", index=False)

In [72]:
men_rf_importance_df.to_csv("../data/processed/rf_feature_importance_men.csv", index=False)
men_xgb_importance_df.to_csv("../data/processed/xgb_feature_importance_men.csv", index=False)


In [73]:
women_rf_importance_df.to_csv("../data/processed/rf_feature_importance_women.csv", index=False)
women_xgb_importance_df.to_csv("../data/processed/xgb_feature_importance_women.csv", index=False)

#### Updating XGBoost to:
`
    n_estimators=500,
    max_depth=3,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
`
So now XGBoost has a lower learning rate and more trees, so it will hopefully generalize better